# YahtzeeRL — Self-Play Training Notebook

Two-player Yahtzee self-play with JAX, Flax, RLax, and MCTS.

**Runtime:** Colab GPU  
**Pipeline:** GPU check -> Drive mount -> local repo copy -> install -> tests -> smoke train -> full train

## 0 — Environment Setup

In [ ]:
# -- Verify GPU -------------------------------------------------------------
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import subprocess

gpu_info = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
    text=True,
).strip()
assert gpu_info, 'No GPU detected. In Colab: Runtime -> Change runtime type -> GPU.'
print(f'GPU: {gpu_info}')

In [ ]:
# -- Mount Google Drive for persistent repo/checkpoints ---------------------
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/yahtzeeRL'  # change if needed
CHECKPOINT_ROOT = f'{DRIVE_ROOT}/checkpoints'
!mkdir -p {CHECKPOINT_ROOT}

In [ ]:
# -- Copy repo from Drive to local Colab disk -------------------------------
# Running from /content is faster and avoids Drive filesystem quirks during training.
import os
import shutil

REPO_DIR = '/content/yahtzeeRL'
REFRESH_LOCAL_COPY = True

# OPTION A: Clone from git instead of Drive.
# REPO_URL = 'https://github.com/YOUR_USER/yahtzeeRL.git'
# !git clone {REPO_URL} {REPO_DIR}

# OPTION B: Copy from Drive.
if REFRESH_LOCAL_COPY and os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print(f'Removed stale local copy at {REPO_DIR}')

if not os.path.isdir(REPO_DIR):
    if os.path.isdir(DRIVE_ROOT):
        shutil.copytree(
            DRIVE_ROOT,
            REPO_DIR,
            ignore=shutil.ignore_patterns('venv', '.venv', '__pycache__', '.pytest_cache', 'checkpoints'),
        )
        print(f'Copied repo from {DRIVE_ROOT} to {REPO_DIR}')
    else:
        raise FileNotFoundError(f'No repo found at {DRIVE_ROOT}. Copy yahtzeeRL there or use git clone above.')

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
!ls -la

In [ ]:
# -- Install dependencies ---------------------------------------------------
# Colab commonly exposes CUDA 12 runtimes. If this fails on a newer runtime,
# change jax[cuda12] to the CUDA extra shown in the current JAX install docs.
!pip install -q -U pip
!pip install -q -U "jax[cuda12]"
!pip install -q -e .[dev]

import jax

print(f'JAX devices: {jax.devices()}')
assert any('gpu' in str(d).lower() or 'cuda' in str(d).lower() for d in jax.devices()), \
    'JAX is not using the GPU. Restart the runtime after install, or check the CUDA JAX wheel.'

## 1 — Sanity Checks

In [ ]:
# -- Run tests --------------------------------------------------------------
!python -m pytest -q

In [ ]:
# -- Tiny MCTS/self-play/training smoke test -------------------------------
!python -m yahtzee_rl.train \
  --steps 1 \
  --batch-size 1 \
  --num-simulations 1 \
  --checkpoint-dir {CHECKPOINT_ROOT}/smoke

## 2 — Training

In [ ]:
from yahtzee_rl.train import TrainConfig, train

config = TrainConfig(
    steps=1000,
    batch_size=64,
    num_simulations=32,
    hidden_dim=256,
    checkpoint_dir=f'{CHECKPOINT_ROOT}/colab_run',
    checkpoint_every=100,
    log_every=10,
)

state = train(config)

For a quick iteration run, lower the config to `steps=10`, `batch_size=4`, and `num_simulations=4`. Checkpoints are written to Drive under `yahtzeeRL/checkpoints/`.